<center><h1>Satish_Manvika_HW4</h1></center>
<br>
Name: Manvika Satish
<br>
Github Username: manvsatish 
<br>
USC ID: 5582061880

## 1. Time Series Classification Part 1: Feature Creation/Extraction (HW3 Rerun)

### (a) Obtain Data

Import packages

In [2]:
import io
import re
import warnings
import zipfile
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm

from sklearn.base import clone
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, label_binarize

In [3]:
RANDOM_STATE = 552
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)

Get the AReM Data Set

In [5]:
zip_path = Path("../data/AReM.zip") #relative path
print("ZIP file exists:", zip_path.exists())
print("Relative path:", zip_path)
assert zip_path.exists(), f"Data file not found at {zip_path}"

#valid csv files
with zipfile.ZipFile(zip_path, "r") as archive:
    csv_files = sorted(name for name in archive.namelist() if name.endswith(".csv") and "__MACOSX" not in name)

print("Number of CSV files:", len(csv_files))
print("First five files:")
for name in csv_files[:5]:
    print(name)

assert len(csv_files) == 88, (f"Expected 88 CSV files, found {len(csv_files)}")

ZIP file exists: True
Relative path: ../data/AReM.zip
Number of CSV files: 88
First five files:
AReM/bending1/dataset1.csv
AReM/bending1/dataset2.csv
AReM/bending1/dataset3.csv
AReM/bending1/dataset4.csv
AReM/bending1/dataset5.csv


In [6]:
#col names used by every cleaned arem instance
column_names = ["time", "avg_rss12", "var_rss12", "avg_rss13", "var_rss13", "avg_rss23", "var_rss23",]

# should contain timestamps from 0 to 119750 in increments of 250 = 480 obs
expected_times = np.arange(0, 120000, 250)

#read and clean one csv file directly from zip
def read_activity_csv(archive, member_name):
    #deode the file without extracting it to the filesystem
    text = archive.read(member_name).decode("utf-8", errors="replace",)
    rows = []

    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue

        #most files are comma-delimited, bending2/dataset4.csv is whitespace-delimited.
        parts = line.split(",") if "," in line else line.split()
        parts = [part.strip() for part in parts if part.strip()] # remove empty fields produced by trailing commas

        # ignore rows that do not contain all seven required values.
        if len(parts) < 7:
            continue
        try:
            rows.append([float(value) for value in parts[:7]])
        except ValueError:
            continue # ignore any remaining nonnumeric header row

    data = pd.DataFrame(rows, columns=column_names)
    original_rows = len(data)

    # standard timestamp type and combine duplicate timestamps if any are there
    data["time"] = data["time"].astype(int)
    data = data.groupby("time", as_index=False).mean()

    #find missing expected timestamps before fixing
    missing_times = sorted(set(expected_times) - set(data["time"]))
    needs_repair = (len(data) != len(expected_times) or not np.array_equal(data["time"].to_numpy(), expected_times,))

    # sitting/dataset8.csv is missing timestamp 13500
    # restore missing observations using linear interpolation.
    if needs_repair:
        data = (data.set_index("time").reindex(expected_times).interpolate(method="linear", limit_direction="both",).rename_axis("time").reset_index())

    # check for no missing vals
    assert data.shape == (480, 7), (f"{member_name} has unexpected shape {data.shape}")
    assert not data.isna().any().any(), (f"{member_name} still contains missing values")

    cleaning_info = {"file": member_name, "original_rows": original_rows, "final_rows": len(data), "missing_times": missing_times, "repaired": needs_repair,}
    
    return data, cleaning_info

### (b) Splitting Data

In [7]:
activity_order = ["bending1", "bending2", "cycling", "lying", "sitting", "standing", "walking",]

#sort archive members by activity and numeric ds number
def file_sort_key(member_name):
    activity = Path(member_name).parent.name
    dataset_number = int(re.search(r"dataset(\d+)\.csv$", member_name,).group(1))

    return activity_order.index(activity), dataset_number

# store cleaned activity instance and its split assignment
instances = []
cleaning_records = []

with zipfile.ZipFile(zip_path, "r") as archive:
    for member_name in sorted(csv_files, key=file_sort_key,):
        activity = Path(member_name).parent.name
        dataset_number = int(re.search(r"dataset(\d+)\.csv$", member_name,).group(1))

        # read and clean current
        data, cleaning_info = read_activity_csv(archive, member_name,)

        if activity in {"bending1", "bending2"}:
            split = ("test" if dataset_number in {1, 2} else "train") # datasets 1 and 2 - test data
        else:
            split = ("test" if dataset_number in {1, 2, 3} else "train") # datasets 1, 2, 3 are test data - all other classes

        instances.append({"activity": activity, "dataset": dataset_number, "split": split, "data": data,})
        cleaning_info.update({"activity": activity, "dataset": dataset_number,})
        cleaning_records.append(cleaning_info)

# check if every dataset was loaded exactly once
assert len(instances) == 88

# summ the train/test split by activity
split_summary = (pd.DataFrame(instances).groupby(["activity", "split"]).size().unstack(fill_value=0).sort_index())
split_summary.loc["Total"] = split_summary.sum()

# files that need timestamp repair
cleaning_summary = pd.DataFrame(cleaning_records)
repaired_files = cleaning_summary.loc[cleaning_summary["repaired"], ["activity", "dataset", "original_rows", "final_rows", "missing_times",],]

print("Total instances loaded:", len(instances))
print("\nFiles requiring timestamp repair:")
display(repaired_files)

print("\nTrain/test split:")
display(split_summary)

Total instances loaded: 88

Files requiring timestamp repair:


,activity,dataset,original_rows,final_rows,missing_times
50,sitting,8,479,480,[13500]



Train/test split:


split,test,train
activity,,
bending1,2,5
bending2,2,4
cycling,3,12
lying,3,12
sitting,3,12
standing,3,12
walking,3,12
Total,19,69


### (c) Feature Extraction

In [8]:
sensor_columns = column_names[1:]

#split six time series into ~ equal segments, extract seven time domain fts from every segment
def extract_segmented_features(data, number_of_segments):
    features = {}

    for series_number, column in enumerate(sensor_columns,start=1,):
        segments = np.array_split(data[column].to_numpy(), number_of_segments,)
        for segment_number, segment in enumerate(segments, start=1,):
            feature_prefix = (
                f"series{series_number}_"
                f"segment{segment_number}"
            )

            features[f"{feature_prefix}_min"] = np.min(segment)
            features[f"{feature_prefix}_max"] = np.max(segment)
            features[f"{feature_prefix}_mean"] = np.mean(segment)
            features[f"{feature_prefix}_median"] = np.median(segment)
            features[f"{feature_prefix}_std"] = np.std(segment, ddof=1,)
            features[f"{feature_prefix}_1st_quart"] = np.quantile(segment, 0.25,)
            features[f"{feature_prefix}_3rd_quart"] = np.quantile(segment, 0.75,)
    return features


# feature matrix & labels for l
def build_feature_dataset(instance_list, number_of_segments):
    feature_rows = []
    metadata_rows = []

    for instance_number, instance in enumerate(instance_list, start=1,):
        # extract 42*l time domain features for instance
        feature_rows.append(extract_segmented_features(instance["data"], number_of_segments,))

        # store labels and identifiers separately from num fts
        metadata_rows.append(
            {
                "instance": instance_number,
                "activity": instance["activity"],
                "dataset": instance["dataset"],
                "split": instance["split"],
                "binary_label": int(
                    instance["activity"]
                    in {"bending1", "bending2"}
                ),
            }
        )

    feature_matrix = pd.DataFrame(feature_rows)
    metadata = pd.DataFrame(metadata_rows)

    # six series contributes seven features per segment
    expected_feature_count = 6 * 7 * number_of_segments

    assert feature_matrix.shape == (len(instance_list), expected_feature_count,)
    assert not feature_matrix.isna().any().any()

    return feature_matrix, metadata


# verify extractor for l=1 and 2
features_l1, metadata_l1 = build_feature_dataset(instances, number_of_segments=1,)
features_l2, metadata_l2 = build_feature_dataset(instances, number_of_segments=2,)
print("l=1 feature shape:", features_l1.shape)
print("l=2 feature shape:", features_l2.shape)

# confirm metadata preserves the required split
print("\nSplit counts:")
display(metadata_l1["split"].value_counts())

l=1 feature shape: (88, 42)
l=2 feature shape: (88, 84)

Split counts:


split
train    69
test     19
Name: count, dtype: int64

## 2. Time Series Classification Part 2: Binary and Multiclass Classification

### (a) Binary Classification Using Logistic Regression

#### i. Plots

#### ii. Splitted Plots

#### iii. Time Series Segments

#### iv. Confusion Matrix

#### v. Test Classifier

#### vi. Separation

#### vii. Imbalance

### (b) Binary Classification Using L1-penalized logistic regression

#### i. Time Series Segments

#### ii. Comparison

### (c) Multi-class Classification (The Realistic Case)

#### i. Time Series Segments

#### ii. Naive Bayes

#### ii. Comparison

## 3. ISLR 4.8.3

## 4. ISLR 4.8.7

## 5.1. Extra Practice ISLR 4.8.4

## 5.2. Extra Practice ISLR 4.8.9